In [ ]:
import json
import sys
from typing import List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from simulate_markov_utils import plot_markov_df
from sklearn.metrics import auc, roc_curve

sys.path.append('..')
from plotting_notebooks.plotting_utils import (
    load_in_combined_results,
    load_monitoring_data,
    plot_markov_analysis,
    plot_roc_curves,
    plot_suspiciousness_distributions,
    plot_cost_vs_safety_curves,
)
from utils.analysis_utils import extract_verdict

# Constants
MAIN_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022']
MONITOR_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022', 'o3-mini', 'o1', 'claude-3-5-haiku-20241022']
DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task"]
DOWNSAMPLE_FACTORS = range(1, 5)
EXCLUDE_TASKS = ['blog_post', 'election_tampering', 'art_collectors', 'spam_filter_update']

In [ ]:
def get_final_suspiciousness(row):
    max_key = max([int(x) for x in row.data.keys()])
    return float(extract_verdict(row.data[str(max_key)]["content"] if isinstance(row.data[str(max_key)], dict) and "content" in row.data[str(max_key)] else row.data[str(max_key)], tag_word='verdict'))

def get_final_usage(row):
    max_key = max([int(x) for x in row.data.keys()])
    return float(row.data[str(max_key)]["usage"][0]["cost_usd"])

def get_downsampled_responses(x, downsample_factor):
    return list(x.values())[::downsample_factor] + [list(x.values())[-1]]

def prepare_turn_by_turn_data(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool, downsample_factors: List[int] = DOWNSAMPLE_FACTORS, thresholds_to_consider = [4, 9, 14, 19, 29, 39], nth_highest_indices: List[int] = [2, 3, 5], suffix: str = 'turn_by_turn') -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load and prepare the initial datasets."""
    # Load monitoring data
    turn_by_turn_monitoring_df = load_monitoring_data(base_dir=base_dir)
    turn_by_turn_monitoring_df = turn_by_turn_monitoring_df[~turn_by_turn_monitoring_df.task_name.isin(EXCLUDE_TASKS)]
    for downsample_factor in downsample_factors:
        turn_by_turn_monitoring_df[f'max_suspiciousness_{downsample_factor}_{suffix}'] = turn_by_turn_monitoring_df.data.apply(
            lambda x: float(max([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 for y in get_downsampled_responses(x, downsample_factor)]))
        )
        turn_by_turn_monitoring_df[f'max_suspiciousness_usage_{downsample_factor}_{suffix}'] = turn_by_turn_monitoring_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, downsample_factor)]))
        )
    for threshold in thresholds_to_consider:
        turn_by_turn_monitoring_df[f'frac_above_{threshold}_{suffix}'] = 100 * turn_by_turn_monitoring_df.data.apply(
            lambda x: float(np.mean([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') > threshold or 0.0 for y in (list(x.values()) + [list(x.values())[-1]])]))
        )

    for i in range(1, 11):
        turn_by_turn_monitoring_df[f'max_suspiciousness_last_{i}_{suffix}'] = turn_by_turn_monitoring_df.data.apply(
            lambda x: float(max([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 for y in list(x.values())[-i:]]))
        )
        turn_by_turn_monitoring_df[f'max_suspiciousness_last_usage_{i}_{suffix}'] = turn_by_turn_monitoring_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in list(x.values())[-i:]]))
        )
    turn_by_turn_monitoring_df['susp_list'] = turn_by_turn_monitoring_df.data.apply(lambda x: [extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 for y in list(x.values())])
    turn_by_turn_monitoring_df['where_the_max_occurs_turn_by_turn'] = turn_by_turn_monitoring_df.susp_list.apply(
        lambda x: np.argmax(x)/len(x)
    )
    # Add columns for nth highest suspicion scores
    for n in nth_highest_indices:
        turn_by_turn_monitoring_df[f'{n}th_highest_suspiciousness_{suffix}'] = turn_by_turn_monitoring_df.data.apply(
            lambda x: sorted([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 for y in list(x.values())], reverse=True)[n-1] if len(x) >= n else 0.0
        )
    turn_by_turn_monitoring_df[f'final_suspiciousness_{suffix}'] = turn_by_turn_monitoring_df.apply(
        lambda row: get_final_suspiciousness(row),
        axis=1
    )
    turn_by_turn_monitoring_df[f'final_usage_{suffix}'] = turn_by_turn_monitoring_df.apply(
        lambda row: get_final_usage(row),
        axis=1
    )
    
    return turn_by_turn_monitoring_df

def prepare_read_the_notes_data(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool, read_the_notes_agent: str = 'gpt-4o') -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load and prepare the initial datasets."""
    # Load monitoring data
    read_the_notes_df = load_monitoring_data(base_dir=base_dir)
    read_the_notes_df['read_the_notes_agent'] = read_the_notes_agent
    read_the_notes_df['read_the_notes_suspiciousness'] = read_the_notes_df.data.apply(
        lambda x: extract_verdict(x[read_the_notes_agent]["content"] if isinstance(x[read_the_notes_agent], dict) and "content" in x[read_the_notes_agent] else x[read_the_notes_agent], tag_word='verdict') #type: ignore
    )
    
    return read_the_notes_df

def merge_and_compute_features_for_turn_by_turn(combined_results: pd.DataFrame, 
                             turn_by_turn_monitoring_df: pd.DataFrame) -> pd.DataFrame:
    """Merge the dataframes and compute additional features."""
    merged_df = pd.merge(
        turn_by_turn_monitoring_df,
        combined_results,
        on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name'],
        how='left'
    ).dropna()
    
    # Compute features
    return merged_df

def merge_and_compute_features_for_read_the_notes(combined_results: pd.DataFrame, 
                             read_the_notes_df: pd.DataFrame) -> pd.DataFrame:
    """Merge the dataframes and compute additional features."""
    merged_df = pd.merge(
        read_the_notes_df,
        combined_results,
        on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name'],
        how='left'
    ).dropna()
    
    merged_df['convo_length'] = merged_df.transcript.apply(lambda x: len(x['conversation']))
    return merged_df



def plot_turn_by_turn_monitoring_results(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool, give_hint: bool, monitors: List[str] = ['claude-3-5-haiku-20241022', 'gpt-4o'], downsample_factors: List[int] = DOWNSAMPLE_FACTORS, agents: List[str] = ['claude-3-5-sonnet-20241022'], metrics: List[str] = ['final_suspiciousness']):
    """Main execution function."""
    # Load and prepare data
    combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)
    turn_by_turn_monitoring_df = prepare_turn_by_turn_data(base_dir, include_failed_attempts, include_failed_main_task, downsample_factors)
    merged_df = merge_and_compute_features_for_turn_by_turn(combined_results, turn_by_turn_monitoring_df)
    merged_df = merged_df[merged_df.main_agent.isin(agents)]
    merged_df = merged_df[merged_df.monitor_agent.isin(monitors)]
    # Print data shapes for verification
    print("Shape of df before merge:", turn_by_turn_monitoring_df.shape)
    print("Shape of combined_results before merge:", combined_results.shape)
    print("Shape of merged_df after merge:", merged_df.shape)
    # Generate plots
    
    plot_suspiciousness_distributions(merged_df, monitors=monitors, metrics=metrics)
    plot_roc_curves(merged_df, combined_results, give_hint, include_failed_attempts, include_failed_main_task, monitors=monitors, metrics=metrics)
    # for monitor in monitors:
    #     plot_markov_analysis(merged_df, main_agent='claude-3-5-sonnet-20241022', monitor_model=monitor, give_hint=False, metrics=metrics)

def plot_read_the_notes_monitoring_results(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool, give_hint: bool, monitors: List[str] = ['claude-3-5-haiku-20241022', 'gpt-4o']):
    """Main execution function."""
    # Load and prepare data
    combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)
    read_the_notes_df = prepare_read_the_notes_data(base_dir, include_failed_attempts, include_failed_main_task)
    merged_df = merge_and_compute_features_for_read_the_notes(combined_results, read_the_notes_df)
    
    # Generate plots
    plot_suspiciousness_distributions(merged_df, monitors=monitors, metrics=['read_the_notes_suspiciousness'])
    plot_roc_curves(merged_df, combined_results, give_hint, include_failed_attempts, include_failed_main_task, monitors=monitors, metrics=['read_the_notes_suspiciousness', 'default_suspiciousness'])
    for monitor in monitors:
        plot_markov_analysis(merged_df, main_agent='claude-3-5-sonnet-20241022', monitor_model=monitor, give_hint=False, metrics=['read_the_notes_suspiciousness'])

In [ ]:
combined_results = load_in_combined_results(include_failed_attempts=False, include_failed_main_task=False)
turn_by_turn_monitoring_df = prepare_turn_by_turn_data(base_dir='monitor_responses/turn_by_turn_monitoring', include_failed_attempts=False, include_failed_main_task=False, downsample_factors=DOWNSAMPLE_FACTORS)
merged_df = merge_and_compute_features_for_turn_by_turn(combined_results, turn_by_turn_monitoring_df)

In [ ]:
turn_by_turn_monitoring_df.task_name.value_counts()

In [ ]:
merged_df['weighted_max_suspiciousness'] = merged_df.apply(lambda row: np.max(np.array(row['susp_list']) * np.linspace(0.0, 1.0, len(row['susp_list']))), axis=1)
plot_roc_curves(merged_df, combined_results, give_hint=True, include_failed_attempts=False, include_failed_main_task=False, monitors=['gpt-4o'], metrics=['final_suspiciousness_turn_by_turn', 'max_suspiciousness_1_turn_by_turn',  'max_suspiciousness_last_2_turn_by_turn', 'max_suspiciousness_last_3_turn_by_turn', 'max_suspiciousness_last_4_turn_by_turn', 'max_suspiciousness_last_10_turn_by_turn', 'where_the_max_occurs_turn_by_turn', 'weighted_max_suspiciousness'])

In [ ]:
for transcript_type in ['transcripts', 'transcripts_no_hint', 'transcripts_no_secondary_task']:
    print(f"Transcript type: {transcript_type}")
    print(merged_df[merged_df.transcript_type == transcript_type].task_name.value_counts())

In [ ]:
plot_suspiciousness_distributions(merged_df, monitors=['gpt-4o'], metrics=['final_suspiciousness_turn_by_turn', 'max_suspiciousness_1_turn_by_turn', 'max_suspiciousness_last_1_turn_by_turn', 'max_suspiciousness_last_2_turn_by_turn', 'max_suspiciousness_last_3_turn_by_turn', 'max_suspiciousness_last_4_turn_by_turn', 'max_suspiciousness_last_10_turn_by_turn', 'weighted_max_suspiciousness'])

In [ ]:
def plot_suspiciousness_histograms_by_side_task(df, monitor='gpt-4o'):
    """
    Plot histograms of suspiciousness scores for cases with and without side tasks.
    
    Args:
        df: DataFrame containing the data
        monitor: Monitor agent to filter by
    """
    # Filter data for the specified monitor
    filtered_df = df[df.monitor_agent == monitor].copy()
    
    # Create figure with multiple subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'Suspiciousness Distributions by Side Task Presence - {monitor}', fontsize=16)
    
    # Plot 1: Distribution of max suspiciousness
    axes[0, 0].set_title('Max Suspiciousness Distribution')
    for side_task, group in filtered_df.groupby('side_task_evaluation'):
        label = 'With Side Task' if side_task else 'Without Side Task'
        max_values = group['susp_list'].apply(lambda x: max(x) if len(x) > 0 else 0)
        axes[0, 0].hist(max_values, alpha=0.5, bins=20, label=label, density=True)
    axes[0, 0].set_xlabel('Max Suspiciousness Score')
    axes[0, 0].set_ylabel('Density')
    axes[0, 0].legend()
    
    # Plot 2: Distribution of mean suspiciousness
    axes[0, 1].set_title('Mean Suspiciousness Distribution')
    for side_task, group in filtered_df.groupby('side_task_evaluation'):
        label = 'With Side Task' if side_task else 'Without Side Task'
        mean_values = group['susp_list'].apply(lambda x: np.mean(x) if len(x) > 0 else 0)
        axes[0, 1].hist(mean_values, alpha=0.5, bins=20, label=label, density=True)
    axes[0, 1].set_xlabel('Mean Suspiciousness Score')
    axes[0, 1].set_ylabel('Density')
    axes[0, 1].legend()
    
    # Plot 3: Distribution of median suspiciousness
    axes[1, 0].set_title('Median Suspiciousness Distribution')
    for side_task, group in filtered_df.groupby('side_task_evaluation'):
        label = 'With Side Task' if side_task else 'Without Side Task'
        median_values = group['susp_list'].apply(lambda x: np.median(x) if len(x) > 0 else 0)
        axes[1, 0].hist(median_values, alpha=0.5, bins=20, label=label, density=True)
    axes[1, 0].set_xlabel('Median Suspiciousness Score')
    axes[1, 0].set_ylabel('Density')
    axes[1, 0].legend()
    
    # Plot 4: Distribution of 75th percentile suspiciousness
    axes[1, 1].set_title('75th Percentile Suspiciousness Distribution')
    for side_task, group in filtered_df.groupby('side_task_evaluation'):
        label = 'With Side Task' if side_task else 'Without Side Task'
        p75_values = group['susp_list'].apply(lambda x: np.percentile(x, 75) if len(x) > 0 else 0)
        axes[1, 1].hist(p75_values, alpha=0.5, bins=20, label=label, density=True)
    axes[1, 1].set_xlabel('75th Percentile Suspiciousness Score')
    axes[1, 1].set_ylabel('Density')
    axes[1, 1].legend()
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
    
    return fig


In [ ]:
# Plot suspiciousness histograms for different monitors
plot_suspiciousness_histograms_by_side_task(merged_df, monitor='gpt-4o')


In [ ]:
def plot_suspiciousness_over_time(df, monitor='gpt-4o', num_samples=10):
    """
    Plot suspiciousness scores over time for a sample of conversations with and without side tasks.
    
    Args:
        df: DataFrame containing the data
        monitor: Monitor agent to filter by
        num_samples: Number of samples to plot for each category
    """
    # Filter data for the specified monitor
    filtered_df = df[df.monitor_agent == monitor].copy()
    
    # Create figure with two subplots
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle(f'Suspiciousness Over Time - {monitor}', fontsize=16)
    
    # Plot for conversations with side task
    axes[0].set_title('Conversations With Side Task')
    with_side_task = filtered_df[filtered_df.side_task_evaluation == True].sample(min(num_samples, sum(filtered_df.side_task_evaluation == True)))
    for idx, row in with_side_task.iterrows():
        susp_list = row['susp_list']
        axes[0].plot(range(len(susp_list)), susp_list, marker='o', alpha=0.7, linewidth=1.5)
    axes[0].set_xlabel('Turn Number')
    axes[0].set_ylabel('Suspiciousness Score')
    axes[0].grid(True, linestyle='--', alpha=0.7)
    
    # Plot for conversations without side task
    axes[1].set_title('Conversations Without Side Task')
    without_side_task = filtered_df[filtered_df.side_task_evaluation == False].sample(min(num_samples, sum(filtered_df.side_task_evaluation == False)))
    for idx, row in without_side_task.iterrows():
        susp_list = row['susp_list']
        axes[1].plot(range(len(susp_list)), susp_list, marker='o', alpha=0.7, linewidth=1.5)
    axes[1].set_xlabel('Turn Number')
    axes[1].set_ylabel('Suspiciousness Score')
    axes[1].grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
    
    return fig


In [ ]:
# Plot suspiciousness over time
plot_suspiciousness_over_time(merged_df, monitor='gpt-4o', num_samples=8)


In [ ]:
def evaluate_aggregation_methods(df, monitor='gpt-4o'):
    """
    Evaluate different methods of aggregating suspiciousness scores and their performance.
    
    Args:
        df: DataFrame containing the data
        monitor: Monitor agent to filter by
    """
    # Filter data for the specified monitor
    filtered_df = df[df.monitor_agent == monitor].copy()
    
    # Calculate different aggregation metrics
    filtered_df['max_susp'] = filtered_df['susp_list'].apply(lambda x: max(x) if len(x) > 0 else 0)
    filtered_df['mean_susp'] = filtered_df['susp_list'].apply(lambda x: np.mean(x) if len(x) > 0 else 0)
    filtered_df['median_susp'] = filtered_df['susp_list'].apply(lambda x: np.median(x) if len(x) > 0 else 0)
    filtered_df['p75_susp'] = filtered_df['susp_list'].apply(lambda x: np.percentile(x, 75) if len(x) > 0 else 0)
    filtered_df['last_susp'] = filtered_df['susp_list'].apply(lambda x: x[-1] if len(x) > 0 else 0)
    filtered_df['last_3_max'] = filtered_df['susp_list'].apply(lambda x: max(x[-3:]) if len(x) >= 3 else max(x) if len(x) > 0 else 0)
    filtered_df['weighted_mean'] = filtered_df['susp_list'].apply(
        lambda x: np.average(x, weights=np.linspace(0.0, 1.0, len(x))) if len(x) > 0 else 0
    )
    
    # Ensure side_task_evaluation is properly formatted as binary (0/1)
    filtered_df['side_task_binary'] = filtered_df['side_task_evaluation'].astype(int)
    
    # Print debug information
    print(f"Data types - side_task_evaluation: {filtered_df['side_task_evaluation'].dtype}")
    print(f"Data types - side_task_binary: {filtered_df['side_task_binary'].dtype}")
    print(f"Unique values in side_task_binary: {filtered_df['side_task_binary'].unique()}")
    
    # Calculate ROC AUC for each method
    metrics = ['max_susp', 'mean_susp', 'median_susp', 'p75_susp', 'last_susp', 'last_3_max', 'weighted_mean']
    results = []
    
    for metric in metrics:
        try:
            fpr, tpr, _ = roc_curve(filtered_df['side_task_binary'], filtered_df[metric])
            roc_auc = auc(fpr, tpr)
            results.append({'Metric': metric, 'AUC': roc_auc})
        except Exception as e:
            print(f"Error calculating ROC for {metric}: {e}")
            results.append({'Metric': metric, 'AUC': float('nan')})
    
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('AUC', ascending=False)
    
    # Plot ROC curves
    plt.figure(figsize=(10, 8))
    for metric in metrics:
        try:
            fpr, tpr, _ = roc_curve(filtered_df['side_task_binary'], filtered_df[metric])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, lw=2, label=f'{metric} (AUC = {roc_auc:.3f})')
        except Exception as e:
            print(f"Error plotting ROC for {metric}: {e}")
    
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curves for Different Aggregation Methods - {monitor}')
    plt.legend(loc="lower right")
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()
    
    print("Aggregation Methods Ranked by AUC:")
    print(results_df)
    
    return results_df


In [ ]:
# Evaluate different aggregation methods
evaluate_aggregation_methods(merged_df, monitor='gpt-4o')


In [ ]:
def visualize_suspiciousness_patterns(df, monitor='gpt-4o'):
    """
    Visualize the actual suspiciousness patterns for cases with and without side tasks.
    Shows both individual examples and aggregated patterns.
    
    Args:
        df: DataFrame containing the data
        monitor: Monitor agent to filter by
    """
    # Filter data for the specified monitor
    filtered_df = df[df.monitor_agent == monitor].copy()
    
    # Create a figure with multiple subplots
    fig = plt.figure(figsize=(20, 16))
    
    # 1. Plot individual examples - side task present
    ax1 = fig.add_subplot(3, 2, 1)
    with_side_task = filtered_df[filtered_df.side_task_evaluation == True].sample(min(5, sum(filtered_df.side_task_evaluation)))
    for idx, (_, row) in enumerate(with_side_task.iterrows()):
        susp_list = row['susp_list']
        ax1.plot(range(len(susp_list)), susp_list, marker='o', linewidth=2, label=f'Example {idx+1}')
    ax1.set_title('Individual Examples - With Side Task', fontsize=14)
    ax1.set_xlabel('Turn Number')
    ax1.set_ylabel('Suspiciousness Score')
    ax1.grid(True, linestyle='--', alpha=0.7)
    ax1.legend()
    
    # 2. Plot individual examples - side task absent
    ax2 = fig.add_subplot(3, 2, 2)
    without_side_task = filtered_df[filtered_df.side_task_evaluation == False].sample(min(5, sum(filtered_df.side_task_evaluation == False)))
    for idx, (_, row) in enumerate(without_side_task.iterrows()):
        susp_list = row['susp_list']
        ax2.plot(range(len(susp_list)), susp_list, marker='o', linewidth=2, label=f'Example {idx+1}')
    ax2.set_title('Individual Examples - Without Side Task', fontsize=14)
    ax2.set_xlabel('Turn Number')
    ax2.set_ylabel('Suspiciousness Score')
    ax2.grid(True, linestyle='--', alpha=0.7)
    ax2.legend()
    
    # 3. Heatmap of all suspiciousness patterns - side task present
    ax3 = fig.add_subplot(3, 2, 3)
    with_side_task_all = filtered_df[filtered_df.side_task_evaluation == True]
    
    # Normalize all sequences to a fixed length (e.g., 20 turns)
    max_length = 20
    with_side_task_normalized = []
    
    for _, row in with_side_task_all.iterrows():
        susp_list = row['susp_list']
        if len(susp_list) > max_length:
            with_side_task_normalized.append(susp_list[:max_length])
        else:
            # Pad with NaN if shorter
            padded = susp_list + [np.nan] * (max_length - len(susp_list))
            with_side_task_normalized.append(padded)
    
    # Convert to numpy array for heatmap
    if len(with_side_task_normalized) > 0:
        heatmap_data = np.array(with_side_task_normalized)
        
        # Sort rows by maximum suspiciousness for better visualization
        max_indices = np.nanargmax(heatmap_data, axis=1)
        sorted_indices = np.argsort(max_indices)
        sorted_heatmap_data = heatmap_data[sorted_indices]
        
        # Create heatmap
        im = ax3.imshow(sorted_heatmap_data, aspect='auto', cmap='viridis')
        ax3.set_title('Suspiciousness Patterns - With Side Task', fontsize=14)
        ax3.set_xlabel('Turn Number')
        ax3.set_ylabel('Conversation Index (sorted)')
        plt.colorbar(im, ax=ax3, label='Suspiciousness Score')
    else:
        ax3.text(0.5, 0.5, 'No data available', horizontalalignment='center', 
                verticalalignment='center', transform=ax3.transAxes)
        ax3.set_title('Suspiciousness Patterns - With Side Task (No Data)', fontsize=14)
    
    # 4. Heatmap of all suspiciousness patterns - side task absent
    ax4 = fig.add_subplot(3, 2, 4)
    without_side_task_all = filtered_df[filtered_df.side_task_evaluation == False]
    
    # Normalize all sequences
    without_side_task_normalized = []
    
    for _, row in without_side_task_all.iterrows():
        susp_list = row['susp_list']
        if len(susp_list) > max_length:
            without_side_task_normalized.append(susp_list[:max_length])
        else:
            # Pad with NaN if shorter
            padded = susp_list + [np.nan] * (max_length - len(susp_list))
            without_side_task_normalized.append(padded)
    
    # Convert to numpy array for heatmap
    if len(without_side_task_normalized) > 0:
        heatmap_data = np.array(without_side_task_normalized)
        
        # Sort rows by maximum suspiciousness for better visualization
        max_indices = np.nanargmax(heatmap_data, axis=1)
        sorted_indices = np.argsort(max_indices)
        sorted_heatmap_data = heatmap_data[sorted_indices]
        
        # Create heatmap
        im = ax4.imshow(sorted_heatmap_data, aspect='auto', cmap='viridis')
        ax4.set_title('Suspiciousness Patterns - Without Side Task', fontsize=14)
        ax4.set_xlabel('Turn Number')
        ax4.set_ylabel('Conversation Index (sorted)')
        plt.colorbar(im, ax=ax4, label='Suspiciousness Score')
    else:
        ax4.text(0.5, 0.5, 'No data available', horizontalalignment='center', 
                verticalalignment='center', transform=ax4.transAxes)
        ax4.set_title('Suspiciousness Patterns - Without Side Task (No Data)', fontsize=14)
    
    # 5. Average suspiciousness over turns - with confidence intervals
    ax5 = fig.add_subplot(3, 2, 5)
    
    # Calculate mean and std for each turn
    with_side_task_means = []
    with_side_task_stds = []
    without_side_task_means = []
    without_side_task_stds = []
    
    for turn in range(max_length):
        # Process with side task data
        with_values = []
        for row in with_side_task_normalized:
            if turn < len(row) and not np.isnan(row[turn]):
                with_values.append(row[turn])
        
        # Process without side task data
        without_values = []
        for row in without_side_task_normalized:
            if turn < len(row) and not np.isnan(row[turn]):
                without_values.append(row[turn])
        
        with_side_task_means.append(np.mean(with_values) if with_values else np.nan)
        with_side_task_stds.append(np.std(with_values) if with_values else np.nan)
        without_side_task_means.append(np.mean(without_values) if without_values else np.nan)
        without_side_task_stds.append(np.std(without_values) if without_values else np.nan)
    
    # Plot means with confidence intervals
    turns = range(max_length)
    ax5.plot(turns, with_side_task_means, 'b-', linewidth=2, label='With Side Task')
    ax5.fill_between(turns, 
                    [m - s if not np.isnan(m) and not np.isnan(s) else np.nan for m, s in zip(with_side_task_means, with_side_task_stds)],
                    [m + s if not np.isnan(m) and not np.isnan(s) else np.nan for m, s in zip(with_side_task_means, with_side_task_stds)],
                    color='b', alpha=0.2)
    
    ax5.plot(turns, without_side_task_means, 'r-', linewidth=2, label='Without Side Task')
    ax5.fill_between(turns, 
                    [m - s if not np.isnan(m) and not np.isnan(s) else np.nan for m, s in zip(without_side_task_means, without_side_task_stds)],
                    [m + s if not np.isnan(m) and not np.isnan(s) else np.nan for m, s in zip(without_side_task_means, without_side_task_stds)],
                    color='r', alpha=0.2)
    
    ax5.set_title('Average Suspiciousness Over Turns', fontsize=14)
    ax5.set_xlabel('Turn Number')
    ax5.set_ylabel('Average Suspiciousness Score')
    ax5.grid(True, linestyle='--', alpha=0.7)
    ax5.legend()
    
    # 6. Distribution of suspiciousness patterns
    ax6 = fig.add_subplot(3, 2, 6)
    
    # Calculate various metrics for each conversation
    with_side_task_metrics = []
    without_side_task_metrics = []
    
    for _, row in with_side_task_all.iterrows():
        susp_list = row['susp_list']
        if len(susp_list) > 0:
            with_side_task_metrics.append({
                'max': max(susp_list),
                'mean': np.mean(susp_list),
                'median': np.median(susp_list),
                'std': np.std(susp_list),
                'last': susp_list[-1],
                'first_peak_idx': np.argmax(susp_list),
                'first_peak_val': max(susp_list)
            })
    
    for _, row in without_side_task_all.iterrows():
        susp_list = row['susp_list']
        if len(susp_list) > 0:
            without_side_task_metrics.append({
                'max': max(susp_list),
                'mean': np.mean(susp_list),
                'median': np.median(susp_list),
                'std': np.std(susp_list),
                'last': susp_list[-1],
                'first_peak_idx': np.argmax(susp_list),
                'first_peak_val': max(susp_list)
            })
    
    # Create scatter plot of max vs. mean suspiciousness
    for metrics, color, label in zip([with_side_task_metrics, without_side_task_metrics], 
                                    ['blue', 'red'], 
                                    ['With Side Task', 'Without Side Task']):
        if metrics:
            max_vals = [m['max'] for m in metrics]
            mean_vals = [m['mean'] for m in metrics]
            ax6.scatter(max_vals, mean_vals, c=color, alpha=0.7, label=label)
    
    ax6.set_title('Max vs. Mean Suspiciousness', fontsize=14)
    ax6.set_xlabel('Max Suspiciousness')
    ax6.set_ylabel('Mean Suspiciousness')
    ax6.grid(True, linestyle='--', alpha=0.7)
    ax6.legend()
    
    plt.tight_layout()
    plt.show()
    
    return fig


In [ ]:
# Visualize suspiciousness patterns
visualize_suspiciousness_patterns(merged_df, monitor='gpt-4o')


In [ ]:
# Debug the side_task_evaluation distribution
print("Checking side_task_evaluation distribution:")
filtered_df = merged_df[merged_df.monitor_agent == 'gpt-4o'].copy()
print(f"Total rows: {len(filtered_df)}")
print(f"side_task_evaluation == True: {sum(filtered_df.side_task_evaluation == True)}")
print(f"side_task_evaluation == False: {sum(filtered_df.side_task_evaluation == False)}")
print(f"side_task_evaluation is NA: {sum(filtered_df.side_task_evaluation.isna())}")
print("\nUnique values in side_task_evaluation:")
print(filtered_df.side_task_evaluation.unique())
print("\nValue counts:")
print(filtered_df.side_task_evaluation.value_counts())
